In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import urllib.request
import copy
import json
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, mean_squared_error
from tqdm import tqdm

# ==========================================
# 1. 实验配置 (针对 Math2015)
# ==========================================

class Config:
    def __init__(self):
        self.num_students = 0
        self.num_questions = 0
        self.num_concepts = 0

        # 模型参数
        self.embed_dim = 64
        self.seq_len = 100
        self.tcan_layers = 2
        self.kernel_size = 3
        self.dropout = 0.2

        # 训练参数
        self.batch_size = 64
        self.epochs = 5  # Math2015数据量较大，演示时设为5，实际建议10+
        self.lr = 0.001
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # 数据集路径
        self.data_dir = "./data"
        self.dataset_name = "math2015"
        self.fallback_url = ""

        # --- 消融实验控制开关 (动态修改) ---
        self.use_student_interaction = True
        self.use_knowledge_concept = True

# ==========================================
# 2. 数据处理模块 (Math2015 专用适配版)
# ==========================================

class AssistDataset(Dataset):
    def __init__(self, q_ids, labels, s_ids, max_len):
        self.q_ids = q_ids
        self.labels = labels
        self.s_ids = s_ids
        self.max_len = max_len

    def __len__(self):
        return len(self.q_ids)

    def __getitem__(self, idx):
        q_seq = self.q_ids[idx]
        r_seq = self.labels[idx]
        s_id = self.s_ids[idx]

        seq_len = len(q_seq)
        if seq_len >= self.max_len:
            q_seq = q_seq[-self.max_len:]
            r_seq = r_seq[-self.max_len:]
            mask = np.ones(self.max_len, dtype=np.float32)
        else:
            pad_len = self.max_len - seq_len
            q_seq = np.pad(q_seq, (pad_len, 0), 'constant', constant_values=0)
            r_seq = np.pad(r_seq, (pad_len, 0), 'constant', constant_values=0)
            mask = np.concatenate([np.zeros(pad_len), np.ones(seq_len)]).astype(np.float32)

        return (
            torch.tensor(q_seq, dtype=torch.long),
            torch.tensor(r_seq, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(s_id, dtype=torch.long)
        )

class DataProcessor:
    def __init__(self, config):
        self.config = config
        if not os.path.exists(config.data_dir):
            os.makedirs(config.data_dir)

    def _install_edudata(self):
        try:
            from EduData import get_data
            return True
        except ImportError:
            print("正在尝试自动安装 EduData 库...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "EduData"])
                print("EduData 安装成功！")
                return True
            except:
                return False

    def download_data(self):
        existing_file = self._find_data_file(self.config.data_dir)
        if existing_file:
            print(f"发现已存在的数据文件: {existing_file}")
            return existing_file

        print(f"正在准备下载数据集: {self.config.dataset_name}")
        try:
            if self._install_edudata():
                from EduData import get_data
                # EduData 下载 math2015 通常会自动解压 RAR
                get_data(self.config.dataset_name, self.config.data_dir)
        except Exception as e:
            print(f"EduData 下载过程出错: {e}")

        return self._find_data_file(self.config.data_dir)

    def _find_data_file(self, root_dir):
        # Math2015 可能是 txt, csv, tsv, json
        valid_extensions = ['.csv', '.txt', '.tsv', '.dat', '.json']
        candidates = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.startswith('.'): continue
                if any(file.lower().endswith(ext) for ext in valid_extensions):
                    if 'q.txt' in file.lower(): continue # 排除 Q 矩阵文件
                    full_path = os.path.join(root, file)
                    candidates.append(full_path)

        if not candidates: return None
        # 优先 data.txt
        candidates.sort(key=lambda x: (not ('data' in os.path.basename(x).lower()), -os.path.getsize(x)))
        return candidates[0]

    def process(self):
        file_path = self.download_data()
        if not file_path: raise FileNotFoundError("未找到有效数据文件。")

        print(f"读取数据: {file_path} ...")

        # 1. 尝试读取
        try:
            df = pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8')
        except:
            try:
                df = pd.read_csv(file_path, sep=None, engine='python', encoding='latin-1')
            except:
                # 尝试无表头读取
                df = pd.read_csv(file_path, sep=None, engine='python', header=None)

        # 2. 列名映射与矩阵检测
        cols = [str(c).lower() for c in df.columns]
        col_map = {str(c).lower(): c for c in df.columns}

        user_col = next((col_map[c] for c in cols if 'user' in c or 'student' in c or 'uid' in c), None)
        item_col = next((col_map[c] for c in cols if 'problem' in c or 'item' in c or 'pid' in c), None)
        skill_col = next((col_map[c] for c in cols if 'skill' in c or 'concept' in c or 'kc' in c), None)
        correct_col = next((col_map[c] for c in cols if 'correct' in c or 'score' in c or 'answer' in c), None)
        order_col = next((col_map[c] for c in cols if 'order' in c or 'time' in c), None)

        # --- 矩阵格式转换 (Wide to Long) ---
        if user_col is None or item_col is None:
            if df.shape[1] > 5: # 假设是 User-Item 矩阵
                print(">>> 检测到矩阵格式，正在转换...")
                # 重新读取无表头
                try:
                    df = pd.read_csv(file_path, sep=None, engine='python', header=None)
                except: pass

                df.index.name = 'user_id'
                df.columns.name = 'item_id'
                df = df.reset_index().melt(id_vars='user_id', var_name='item_id', value_name='correct')

                user_col = 'user_id'
                item_col = 'item_id'
                correct_col = 'correct'
                df.dropna(subset=[correct_col], inplace=True)

        if not (user_col and item_col and correct_col):
            raise ValueError("关键列缺失且无法自动识别为矩阵格式。")

        # 3. 寻找 Skill 信息 (Q-Matrix)
        if not skill_col:
            data_dir = os.path.dirname(file_path)
            q_file = None
            # 搜索 q.txt
            for root, dirs, files in os.walk(self.config.data_dir):
                for f in files:
                    if 'q.txt' in f.lower() or 'q_matrix' in f.lower():
                        q_file = os.path.join(root, f)
                        break
                if q_file: break

            if q_file and os.path.exists(q_file):
                print(f"加载 Q-Matrix: {q_file}")
                try:
                    q_df = pd.read_csv(q_file, sep=None, engine='python', header=None)
                    item_skill_map = {}
                    for idx, row in q_df.iterrows():
                        skills = row[row == 1].index.tolist()
                        if skills: item_skill_map[idx] = skills[0] # 简化取第一个
                        else: item_skill_map[idx] = 0

                    skill_col = 'skill_mapped'
                    df[skill_col] = df[item_col].map(item_skill_map).fillna(0)
                except:
                    skill_col = 'temp_skill'
                    df[skill_col] = df[item_col]
            else:
                print("未找到 Q-Matrix，使用 ItemID 代替 SkillID")
                skill_col = 'temp_skill'
                df[skill_col] = df[item_col]

        # 清洗
        df.dropna(subset=[skill_col], inplace=True)
        df[correct_col] = pd.to_numeric(df[correct_col], errors='coerce').fillna(0).apply(lambda x: 1 if x>=0.5 else 0)

        # ID Mapping
        def get_map(unique_vals):
            return {val: i+1 for i, val in enumerate(unique_vals)}

        user_map = get_map(df[user_col].unique())
        df['uid'] = df[user_col].map(user_map)
        self.config.num_students = len(user_map) + 1

        prob_map = get_map(df[item_col].unique())
        df['iid'] = df[item_col].map(prob_map)
        self.config.num_questions = len(prob_map) + 1

        skill_map = get_map(df[skill_col].unique())
        df['sid'] = df[skill_col].map(skill_map)
        self.config.num_concepts = len(skill_map) + 1

        print(f"统计: 学生 {self.config.num_students}, 题目 {self.config.num_questions}, 知识点 {self.config.num_concepts}")

        # Difficulty
        item_diff = df.groupby('iid')[correct_col].mean()
        diff_values = np.zeros(self.config.num_questions)
        for iid, diff in item_diff.items():
            diff_values[iid] = diff

        # Q-Matrix
        q_k_relation = np.zeros((self.config.num_questions, self.config.num_concepts))
        tmp_df = df[['iid', 'sid']].drop_duplicates()
        for _, row in tmp_df.iterrows():
            q_k_relation[int(row['iid']), int(row['sid'])] = 1
        row_sums = q_k_relation.sum(axis=1)
        row_sums[row_sums == 0] = 1
        q_k_relation = q_k_relation / row_sums[:, np.newaxis]

        # Sequence
        if order_col: df.sort_values(by=[order_col], inplace=True)

        all_q, all_r, all_s = [], [], []
        # 全量跑太慢，采样演示 (实际跑请注释下一行)
        users = df['uid'].unique()[:1000]
        df_small = df[df['uid'].isin(users)]
        # df_small = df

        for uid, group in tqdm(df_small.groupby('uid')):
            if len(group) < 5: continue
            all_q.append(group['iid'].values)
            all_r.append(group[correct_col].values)
            all_s.append(uid)

        return all_q, all_r, all_s, q_k_relation, diff_values

# ==========================================
# 3. 核心模型 (支持消融)
# ==========================================

class HeteroGraphEmbedding(nn.Module):
    def __init__(self, config, diff_values):
        super(HeteroGraphEmbedding, self).__init__()
        self.config = config
        self.embed_dim = config.embed_dim

        # 1. ID (Always)
        self.question_emb = nn.Embedding(config.num_questions, config.embed_dim, padding_idx=0)

        # 2. Knowledge (Optional)
        if self.config.use_knowledge_concept:
            self.concept_emb = nn.Embedding(config.num_concepts, config.embed_dim, padding_idx=0)

        # 3. Difficulty (Optional)
        if self.config.use_student_interaction:
            self.register_buffer('diff_values', torch.tensor(diff_values, dtype=torch.float32))
            self.diff_proj = nn.Linear(1, config.embed_dim)

    def forward(self, question_ids, q_k_relation):
        q_feat = self.question_emb(question_ids)

        if self.config.use_knowledge_concept:
            k_emb_weight = self.concept_emb.weight
            q_k_agg = torch.matmul(q_k_relation, k_emb_weight)
            q_k_feat = F.embedding(question_ids, q_k_agg, padding_idx=0)
            q_feat = q_feat + q_k_feat

        if self.config.use_student_interaction:
            d_val = self.diff_values[question_ids].unsqueeze(-1)
            d_feat = self.diff_proj(d_val)
            q_feat = q_feat + d_feat

        return q_feat

class TemporalAttention(nn.Module):
    def __init__(self, embed_dim, dropout=0.1):
        super(TemporalAttention, self).__init__()
        self.query = nn.Linear(embed_dim, embed_dim)
        self.key = nn.Linear(embed_dim, embed_dim)
        self.value = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, L, D = x.size()
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(D)
        mask = torch.tril(torch.ones(L, L, device=x.device))
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(self.dropout(attn), V)

class TCANBlock(nn.Module):
    def __init__(self, embed_dim, kernel_size, dilation, dropout):
        super(TCANBlock, self).__init__()
        self.ta = TemporalAttention(embed_dim, dropout)
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(embed_dim, embed_dim, kernel_size, dilation=dilation, padding=self.padding)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        z = self.ta(x)
        z_perm = z.permute(0, 2, 1)
        conv_out = self.conv(z_perm)
        conv_out = conv_out[:, :, :z.size(1)].permute(0, 2, 1)
        out = self.norm(conv_out + residual)
        return self.dropout(self.relu(out))

class HIG_TCAN_CD(nn.Module):
    def __init__(self, config, q_k_relation, diff_values):
        super(HIG_TCAN_CD, self).__init__()
        self.config = config
        self.register_buffer('q_k_relation', torch.tensor(q_k_relation, dtype=torch.float32))

        self.graph_emb = HeteroGraphEmbedding(config, diff_values)
        self.student_emb = nn.Embedding(config.num_students, config.embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(config.embed_dim + 1, config.embed_dim)

        self.tcan_layers = nn.ModuleList([
            TCANBlock(config.embed_dim, config.kernel_size, 2**i, config.dropout)
            for i in range(config.tcan_layers)
        ])

        self.pred_mlp = nn.Sequential(
            nn.Linear(config.embed_dim * 3, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, q_ids, answers, student_ids):
        q_emb = self.graph_emb(q_ids, self.q_k_relation)
        x = torch.cat([q_emb, answers.unsqueeze(-1)], dim=-1)
        x = self.input_proj(x)

        h = x
        for layer in self.tcan_layers:
            h = layer(h)

        s_static = self.student_emb(student_ids).unsqueeze(1).expand(-1, q_ids.size(1), -1)
        return h, q_emb, s_static

# ==========================================
# 4. 训练与评估流程
# ==========================================

def train_and_evaluate(config, q_seqs, r_seqs, s_seqs, q_k_rel, diff_values, exp_name):
    train_q, test_q, train_r, test_r, train_s, test_s = train_test_split(
        q_seqs, r_seqs, s_seqs, test_size=0.2, random_state=42
    )

    train_ds = AssistDataset(train_q, train_r, train_s, config.seq_len)
    test_ds = AssistDataset(test_q, test_r, test_s, config.seq_len)
    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

    model = HIG_TCAN_CD(config, q_k_rel, diff_values).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    print(f"\n>>> 正在运行实验: [{exp_name}]")
    best_results = {"AUC": 0.0, "ACC": 0.0, "RMSE": 1.0}

    for epoch in range(config.epochs):
        model.train()
        for q, r, mask, s in train_loader:
            q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
            optimizer.zero_grad()
            h, q_emb, s_static = model(q, r, s)

            h_pred = h[:, :-1, :]
            q_next = q_emb[:, 1:, :]
            s_next = s_static[:, 1:, :]

            feat = torch.cat([h_pred, q_next, s_next], dim=-1)
            preds = model.pred_mlp(feat).squeeze(-1)

            target = r[:, 1:]
            mask_t = mask[:, 1:]

            loss = F.binary_cross_entropy(preds, target, reduction='none')
            loss = (loss * mask_t).sum() / (mask_t.sum() + 1e-8)
            loss.backward()
            optimizer.step()

        scheduler.step()

        # Evaluation
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for q, r, mask, s in test_loader:
                q, r, mask, s = q.to(config.device), r.to(config.device), mask.to(config.device), s.to(config.device)
                h, q_emb, s_static = model(q, r, s)

                h_pred = h[:, :-1, :]
                q_next = q_emb[:, 1:, :]
                s_next = s_static[:, 1:, :]

                feat = torch.cat([h_pred, q_next, s_next], dim=-1)
                preds = model.pred_mlp(feat).squeeze(-1)

                valid = mask[:, 1:].bool()
                if valid.sum() == 0: continue

                all_preds.append(preds[valid].cpu().numpy())
                all_targets.append(r[:, 1:][valid].cpu().numpy())

        if len(all_preds) > 0:
            y_pred = np.concatenate(all_preds)
            y_true = np.concatenate(all_targets)

            auc = roc_auc_score(y_true, y_pred)
            acc = accuracy_score(y_true, (y_pred >= 0.5).astype(int))
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))

            if auc > best_results["AUC"]:
                best_results = {"AUC": auc, "ACC": acc, "RMSE": rmse}

            if (epoch + 1) % 1 == 0: # Print every epoch
                print(f"    Epoch {epoch+1}: ACC={acc:.4f}, RMSE={rmse:.4f}, AUC={auc:.4f}")

    print(f"    [{exp_name}] 最佳结果: ACC={best_results['ACC']:.4f}, RMSE={best_results['RMSE']:.4f}, AUC={best_results['AUC']:.4f}")
    return best_results

def run_ablation_experiments():
    print("====== 准备数据 (Data Preparation: Math2015) ======")
    base_config = Config()
    processor = DataProcessor(base_config)
    try:
        q_seqs, r_seqs, s_seqs, q_k_rel, diff_values = processor.process()
    except Exception as e:
        print(f"数据处理失败: {e}")
        return

    # 定义四组消融实验
    experiments = [
        ("(1) 无异质聚合 (Baseline)", False, False),
        ("(2) 仅学生-试题交互 (Difficulty)", True, False),
        ("(3) 仅试题-知识点交互 (Knowledge)", False, True),
        ("(4) 完整异质聚合 (Full Model)", True, True)
    ]

    final_results = {}

    print("\n====== 开始消融实验 (Ablation Study) ======")
    for name, use_si, use_kc in experiments:
        curr_config = copy.deepcopy(base_config)
        curr_config.use_student_interaction = use_si
        curr_config.use_knowledge_concept = use_kc

        res = train_and_evaluate(curr_config, q_seqs, r_seqs, s_seqs, q_k_rel, diff_values, name)
        final_results[name] = res

    print("\n====== 消融实验总结 (Summary) ======")
    print(f"{'Experiment Name':<40} | {'ACC':<8} | {'RMSE':<8} | {'AUC':<8}")
    print("-" * 75)
    for name, metrics in final_results.items():
        print(f"{name:<40} | {metrics['ACC']:.4f}   | {metrics['RMSE']:.4f}   | {metrics['AUC']:.4f}")

if __name__ == "__main__":
    run_ablation_experiments()

====== 准备数据 (Data Preparation: Math2015) ======
正在准备下载数据集: math2015
正在尝试自动安装 EduData 库...
EduData 安装成功！


downloader, INFO http://staff.ustc.edu.cn/~qiliuql/data/math2015.rar is saved as data/math2015.rar


downloader, INFO data/math2015.rar is unrar to data/math2015



读取数据: ./data/math2015/math2015/Math1/data.txt ...
>>> 检测到矩阵格式，正在转换...
加载 Q-Matrix: ./data/math2015/math2015/Math1/q.txt
统计: 学生 4210, 题目 21, 知识点 7


100%|██████████| 1000/1000 [00:00<00:00, 15388.78it/s]


====== 开始消融实验 (Ablation Study) ======



>>> 正在运行实验: [(1) 无异质聚合 (Baseline)]
    Epoch 1: ACC=0.7230, RMSE=0.4385, AUC=0.7773
    Epoch 2: ACC=0.7282, RMSE=0.4299, AUC=0.7868
    Epoch 3: ACC=0.7272, RMSE=0.4285, AUC=0.7877
    Epoch 4: ACC=0.7295, RMSE=0.4278, AUC=0.7889
    Epoch 5: ACC=0.7295, RMSE=0.4278, AUC=0.7888
    [(1) 无异质聚合 (Baseline)] 最佳结果: ACC=0.7295, RMSE=0.4278, AUC=0.7889

>>> 正在运行实验: [(2) 仅学生-试题交互 (Difficulty)]
    Epoch 1: ACC=0.7245, RMSE=0.4365, AUC=0.7831
    Epoch 2: ACC=0.7192, RMSE=0.4333, AUC=0.7901
    Epoch 3: ACC=0.7183, RMSE=0.4306, AUC=0.7894
    Epoch 4: ACC=0.7255, RMSE=0.4276, AUC=0.7894
    Epoch 5: ACC=0.7268, RMSE=0.4279, AUC=0.7897
    [(2) 仅学生-试题交互 (Difficulty)] 最佳结果: ACC=0.7192, RMSE=0.4333, AUC=0.7901

>>> 正在运行实验: [(3) 仅试题-知识点交互 (Knowledge)]
    Epoch 1: ACC=0.7190, RMSE=0.4418, AUC=0.7710
    Epoch 2: ACC=0.7260, RMSE=0.4315, AUC=0.7853
    Epoch 3: ACC=0.7235, RMSE=0.4304, AUC=0.7873
    Epoch 4: ACC=0.7230, RMSE=0.4296, AUC=0.7883
    Epoch 5: ACC=0.7235, RMSE=0.4290, AUC=0.7878
    